# OpenDecision — Phase 4A.0/4A.1 StateQuery Workbench

**Purpose:** build a versioned state-centric multi-question corpus, then test whether a small learned query network can interrogate a **frozen, state-only Qwen3.5-4B representation** well enough to match the existing candidate-conditioned OpenDecision reference.

**Workbench version:** `4a.0.2`  
**Starting profile:** `a047d6802c3f06f085b8`  
**Frozen backbone:** `Qwen/Qwen3.5-4B-Base@1001bb4d826a52d1f399e183466143f4da7b741b`  
**Reference bundle:** `4d9ffdee0aea5c71c666d0feae372cffe79a05934aedee2245012e3a53c23332`

This notebook is deliberately split into three restartable stages:

1. **4A.0 — corpus:** ingest, normalize, audit, split by source state/component, and lock the data contract.
2. **4A.1 — representation probe:** cache frozen Qwen state sequences and compare B1/B2 query architectures.
3. **Selection/export:** calibrate on non-final partitions, freeze a model lock, and only then permit one final evaluation.

The default configuration is a **small preparation smoke run** with GPU work disabled. Change the clearly labeled configuration cell only after its printed contract matches the intended study.


## Method commitments adopted from SemIf

The benchmark discipline here is informed by SemIf's pinned Phase 1 method at commit [`ca3ba65f`](https://github.com/TheoLeeCJ/SemIf/blob/ca3ba65f142967030ecb453346e94d6f476a69df/docs/METHOD.md). This notebook adopts the parts that transfer cleanly to a training study:

- freeze IDs, prompts/renderers, labels, split rules, model revisions, metrics, and timing scope **before** model selection;
- split by source state and connected/upstream group, never by expanded question row;
- average questions within a state before averaging states, so a 17-question contract is not worth 17 single-question states;
- keep unlike tasks and sources separate instead of hiding them in one pooled accuracy;
- use full-denominator accuracy, balanced accuracy, macro F1, NLL/Brier/ECE, semantic-none metrics, and state-group bootstrap intervals where applicable;
- define output-blind perturbations in advance and align probabilities by semantic option ID;
- report exactly what latency includes and excludes;
- distinguish a typed probability vector from calibrated operational confidence.

SemIf fixtures and third-party raw data are **not** copied into this corpus. Only the methodological controls are reused. The pinned SemIf source-selection manifest is recorded for provenance, not treated as OpenDecision training data.


## 1. Runtime and dependency contract

Use a Colab GPU runtime from the same Torch 2.11 family as Phase 2IJ/3A/3B. The install cell constrains pip so it cannot silently replace the runtime's Torch build.


In [35]:
import sys, subprocess, importlib.metadata as im, json, os, platform
from pathlib import Path

def installed(name):
    try:
        return im.version(name)
    except im.PackageNotFoundError:
        return None

torch_version = installed("torch")
if torch_version is None:
    raise RuntimeError("Select a Colab GPU runtime with PyTorch installed.")

constraint = Path("/content/opendecision_phase4a_torch_constraint.txt")
constraint.write_text(f"torch=={torch_version}\n")

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "-c", str(constraint),
    "transformers==5.17.0",
    "accelerate==1.14.0",
    "safetensors==0.8.0",
    "huggingface-hub",
    "datasets>=4,<5",
    "numpy>=2,<3",
    "scipy>=1.16,<2",
    "pandas>=2,<3",
    "scikit-learn>=1.6,<2",
    "pyarrow>=18,<30",
    "psutil>=5,<8",
    "nbformat>=5.10,<6",
])

if installed("torch") != torch_version:
    raise RuntimeError("Torch changed unexpectedly during dependency installation.")

ENVIRONMENT = {
    "python": sys.version,
    "platform": platform.platform(),
    "packages": {name: installed(name) for name in (
        "torch", "transformers", "accelerate", "safetensors",
        "huggingface-hub", "datasets", "numpy", "scipy",
        "pandas", "scikit-learn", "pyarrow", "psutil",
    )},
}
print(json.dumps(ENVIRONMENT, indent=2))
if not torch_version.startswith("2.11."):
    print("GPU MODEL STAGES WILL BE BLOCKED: the reference environment requires Torch 2.11.x.")


{
  "python": "3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.39",
  "packages": {
    "torch": "2.11.0+cu128",
    "transformers": "5.17.0",
    "accelerate": "1.14.0",
    "safetensors": "0.8.0",
    "huggingface-hub": "1.29.0",
    "datasets": "4.8.5",
    "numpy": "2.1.3",
    "scipy": "1.16.3",
    "pandas": "2.2.3",
    "scikit-learn": "1.6.1",
    "pyarrow": "23.0.1",
    "psutil": "5.9.5"
  }
}


## 2. Mount Drive and read credentials securely

Results are written beside the earlier OpenDecision notebooks. `HF_TOKEN` is read from Colab Secrets and is never serialized into manifests, logs, or exported artifacts. Public assets may work without a token.


In [36]:
from google.colab import drive, userdata

drive.mount("/content/drive")

try:
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = None
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    print("HF_TOKEN loaded from Colab Secrets.")
else:
    print("No HF_TOKEN found; public assets may still load.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
HF_TOKEN loaded from Colab Secrets.


## 3. Declare the study before data or model output

`RUN_MODE="smoke"` limits each enabled source to a few states. Use `RUN_MODE="full"` only after reviewing the source registry and corpus audit. `EXECUTE_GPU=False` makes **Run all** stop after the corpus/contract stage.

Promotion margins are intentionally unset. The final split cannot be opened until they are replaced with reviewed numerical limits and a model-selection lock exists.


In [22]:
import hashlib, random, time, shutil, zipfile, gc
from dataclasses import dataclass, asdict
from datetime import datetime, timezone

WORKBENCH_VERSION = "4a.0.2"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
SEED = 17

COLAB_ROOT = Path("/content/drive/MyDrive/Colab Notebooks")
DRIVE_ROOT = COLAB_ROOT / "OpenDecision_Phase4A_StateQuery_results"
DRIVE_RUN = DRIVE_ROOT / RUN_ID
LOCAL_RUN = Path("/content/opendecision_phase4a") / RUN_ID
for path in (DRIVE_RUN, LOCAL_RUN):
    path.mkdir(parents=True, exist_ok=False)

CFG = {
    "schema": "opendecision-phase4a-config/v1",
    "workbench_version": WORKBENCH_VERSION,
    "run_id": RUN_ID,
    "seed": SEED,
    "run_mode": "full",                 # smoke | full
    "execute_gpu": True,                 # explicit opt-in
    "enabled_sources": ["contractnli", "qasper"],
    "smoke_states_per_source": 16,
    "max_state_tokens": 1024,
    "max_question_tokens": 96,
    "max_candidate_tokens": 64,
    "feature_cache_dtype": "bfloat16",   # confirmatory rerun should also test float32
    "architecture": "B1",                # B0 | B1 | B2 | B2R
    "query_dim": 768,
    "query_heads": 12,
    "query_layers": 2,
    "question_latents": 8,
    "state_refiner_layers": 0,
    "epochs": 12,
    "patience": 3,
    "gradient_accumulation_states": 4,
    "learning_rate": 2e-4,
    "weight_decay": 0.01,
    "evidence_loss_weight": 0.15,
    "reason_loss_weight": 0.10,
    "policy_wrong_cost": 5.0,
    "policy_review_cost": 0.10,
    "open_final": True,
    "promotion_gates": {
        "source_macro_accuracy_delta_min": None,
        "source_macro_nll_delta_max": None,
        "source_macro_ece_delta_max": None,
        "semantic_none_recall_delta_min": None,
        "accepted_error_max": None,
        "minimum_coverage": None,
    },
}

MODEL_CONTRACT = {
    "profile_id": "a047d6802c3f06f085b8",
    "bundle_sha256": "4d9ffdee0aea5c71c666d0feae372cffe79a05934aedee2245012e3a53c23332",
    "model_id": "Qwen/Qwen3.5-4B-Base",
    "revision": "1001bb4d826a52d1f399e183466143f4da7b741b",
    "hidden_size": 2560,
    "parameter_count": 4205751296,
    "reference_repo": "cowWhySo/OpenDecision-Qwen3.5-4B-StateFirst",
}

METHOD_PROVENANCE = {
    "semif_repo": "TheoLeeCJ/SemIf",
    "semif_commit": "ca3ba65f142967030ecb453346e94d6f476a69df",
    "method_blob_sha": "32114ecaaf8d1939fd357ebd736fcd4944161902",
    "source_selection_blob_sha": "d42b34fb07fd82be756b751fc99b76f23569eb77",
    "scope": "methodological controls only; no SemIf fixtures used for training",
}

def canonical_bytes(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=False).encode()

def sha256_obj(obj):
    return hashlib.sha256(canonical_bytes(obj)).hexdigest()

DECLARED_CONTRACT = {
    "config": CFG,
    "model": MODEL_CONTRACT,
    "method_provenance": METHOD_PROVENANCE,
    "environment": ENVIRONMENT,
    "declared_before_data_or_predictions": True,
    "timing_scope": {
        "include": ["tokenization", "host-to-device", "model forward", "query head", "CPU readout"],
        "exclude": ["model download", "model load", "result-file writes", "network", "service queueing"],
    },
}
DECLARED_CONTRACT["sha256"] = sha256_obj(DECLARED_CONTRACT)
for root in (LOCAL_RUN, DRIVE_RUN):
    (root / "DECLARED_CONTRACT.json").write_text(json.dumps(DECLARED_CONTRACT, indent=2))

random.seed(SEED)
print(json.dumps(DECLARED_CONTRACT, indent=2))
print("Drive run:", DRIVE_RUN)


{
  "config": {
    "schema": "opendecision-phase4a-config/v1",
    "workbench_version": "4a.0.2",
    "run_id": "20260921T013558Z",
    "seed": 17,
    "run_mode": "full",
    "execute_gpu": true,
    "enabled_sources": [
      "contractnli",
      "qasper"
    ],
    "smoke_states_per_source": 16,
    "max_state_tokens": 1024,
    "max_question_tokens": 96,
    "max_candidate_tokens": 64,
    "feature_cache_dtype": "bfloat16",
    "architecture": "B1",
    "query_dim": 768,
    "query_heads": 12,
    "query_layers": 2,
    "question_latents": 8,
    "state_refiner_layers": 0,
    "epochs": 12,
    "patience": 3,
    "gradient_accumulation_states": 4,
    "learning_rate": 0.0002,
    "weight_decay": 0.01,
    "evidence_loss_weight": 0.15,
    "reason_loss_weight": 0.1,
    "policy_wrong_cost": 5.0,
    "policy_review_cost": 0.1,
    "open_final": false,
    "promotion_gates": {
      "source_macro_accuracy_delta_min": null,
      "source_macro_nll_delta_max": null,
      "source_macro

## 4. Verify the immutable selected-reference bundle

This is the same bundle identity used by the Phase 3B notebook. Verification happens now; the 4B model is not loaded until the explicit GPU stage.


In [23]:
EXPECTED_BUNDLE = (
    COLAB_ROOT / "OpenDecision_Phase2IJ_results"
    / "2ij_model_selection_screen_v2"
    / "model_bundle_2ij_model_selection_screen_v2.zip"
)
LOCAL_BUNDLE = LOCAL_RUN / "reference_bundle"

def sha256_file(path, chunk=1024 * 1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(chunk):
            h.update(block)
    return h.hexdigest()

def safe_extract(zpath, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(zpath) as zf:
        for member in zf.infolist():
            target = (destination / member.filename).resolve()
            if not target.is_relative_to(destination):
                raise RuntimeError(f"Unsafe ZIP member: {member.filename}")
        zf.extractall(destination)

if not EXPECTED_BUNDLE.is_file():
    raise FileNotFoundError(EXPECTED_BUNDLE)
got_bundle_hash = sha256_file(EXPECTED_BUNDLE)
if got_bundle_hash != MODEL_CONTRACT["bundle_sha256"]:
    raise RuntimeError(f"Reference bundle changed: {got_bundle_hash}")

LOCAL_BUNDLE.mkdir(parents=True, exist_ok=True)
safe_extract(EXPECTED_BUNDLE, LOCAL_BUNDLE)
bundle_manifest = json.loads((LOCAL_BUNDLE / "BUNDLE_MANIFEST.json").read_text())
bundle_profile = json.loads((LOCAL_BUNDLE / "profile.json").read_text())
bundle_protocol = json.loads((LOCAL_BUNDLE / "inference_protocol.json").read_text())

assert bundle_manifest["profile_id"] == MODEL_CONTRACT["profile_id"]
assert bundle_manifest["base"]["id"] == MODEL_CONTRACT["model_id"]
assert bundle_manifest["base"]["revision"] == MODEL_CONTRACT["revision"]
assert bundle_profile["layout"] == "state_first"
for rel, expected in bundle_manifest["files"].items():
    path = LOCAL_BUNDLE / rel
    if not path.is_file() or sha256_file(path) != expected:
        raise RuntimeError(f"Missing or changed bundle member: {rel}")

print("Verified bundle files:", len(bundle_manifest["files"]))
print("Profile:", bundle_profile["profile_id"], bundle_profile["name"])
print("Reference max tokens:", bundle_protocol["max_tokens"])


Verified bundle files: 31
Profile: a047d6802c3f06f085b8 score_summary_seed17
Reference max tokens: 1792


## 5. Phase 4A.0 source registry

The registry distinguishes sources that may be used for model fitting from research/evaluation-only sources. Every Hugging Face source is resolved to a commit SHA at run time **before** loading and that resolution is saved in the corpus manifest.

Initial roles:

| Source | Main role | Default |
|---|---|---|
| ContractNLI | state-once / 17 hypotheses / semantic none / evidence | enabled |
| QASPER | natural paper states / answerability / evidence; pinned Parquet export | enabled |
| MultiRC | independent candidate support (`Noul`) | optional, evaluation-only pending license review |
| SQuAD 2.0 | answerability / insufficient-evidence auxiliary task | optional |
| MASSIVE en-US | dynamic candidate descriptions and held-out intent families | optional |
| QuALITY | long-state four-option evaluation | registered, per-article license review required |
| SummEval | four-rubric Score task | registered, adapter pending |

`NotMentioned` is retained as a named semantic outcome. The training target may use the separate semantic-none slot, but the source label and the reason `insufficient_evidence` remain explicit in the normalized record.

QASPER is read from Hugging Face's auto-converted Parquet branch, resolved to an immutable commit. This avoids executing the legacy `qasper.py` dataset script, which is unsupported by Datasets 4.x, while retaining the published `train`, `validation`, and `test` rows.


In [24]:
from huggingface_hub import HfApi

CONTRACTNLI_COMMIT = "eced6528dd3c1d14d73f9a87df8f7bdbc03126f9"
SOURCE_REGISTRY = {
    "contractnli": {
        "kind": "github_zip",
        "repo": "stanfordnlp/contract-nli",
        "revision": CONTRACTNLI_COMMIT,
        "url": f"https://raw.githubusercontent.com/stanfordnlp/contract-nli/{CONTRACTNLI_COMMIT}/resources/contract-nli.zip",
        "license": "CC-BY-4.0",
        "train_allowed": True,
        "natural": True,
    },
    "qasper": {
        "kind": "huggingface",
        "repo": "allenai/qasper",
        "config": None,
        "loader": "pinned_parquet_export",
        "parquet_ref": "refs/convert/parquet",
        "parquet_config": "qasper",
        "expected_splits": ["train", "validation", "test"],
        "license": "CC-BY-4.0",
        "train_allowed": True,
        "natural": True,
    },
    "multirc": {
        "kind": "huggingface",
        "repo": "aps/super_glue",
        "config": "multirc",
        "license": "custom/research-review-required",
        "train_allowed": False,
        "natural": True,
    },
    "squad2": {
        "kind": "huggingface",
        "repo": "rajpurkar/squad_v2",
        "config": None,
        "license": "CC-BY-SA-4.0",
        "train_allowed": True,
        "natural": True,
    },
    "massive": {
        "kind": "huggingface",
        "repo": "AmazonScience/massive",
        "config": "en-US",
        "license": "CC-BY-4.0",
        "train_allowed": True,
        "natural": True,
    },
    "quality": {
        "kind": "github_zip",
        "repo": "nyu-mll/quality",
        "revision": "main",
        "license": "per-article; evaluation until reviewed",
        "train_allowed": False,
        "natural": True,
        "implemented": False,
    },
    "summeval": {
        "kind": "github",
        "repo": "Yale-LILY/SummEval",
        "license": "review upstream and article provenance before redistribution",
        "train_allowed": False,
        "natural": True,
        "implemented": False,
    },
}

def resolve_hf_source(name):
    spec = SOURCE_REGISTRY[name]
    api = HfApi(token=os.environ.get("HF_TOKEN"))
    info = api.dataset_info(spec["repo"], revision="main")
    resolved = dict(spec)
    resolved["revision"] = info.sha
    resolved["data_revision"] = info.sha
    if spec.get("loader") == "pinned_parquet_export":
        parquet_info = api.dataset_info(spec["repo"], revision=spec["parquet_ref"])
        resolved["parquet_revision"] = parquet_info.sha
        resolved["data_revision"] = parquet_info.sha
    return resolved

RESOLVED_SOURCES = {}
for name in CFG["enabled_sources"]:
    spec = SOURCE_REGISTRY[name]
    RESOLVED_SOURCES[name] = resolve_hf_source(name) if spec["kind"] == "huggingface" else dict(spec)
print(json.dumps(RESOLVED_SOURCES, indent=2))


{
  "contractnli": {
    "kind": "github_zip",
    "repo": "stanfordnlp/contract-nli",
    "revision": "eced6528dd3c1d14d73f9a87df8f7bdbc03126f9",
    "url": "https://raw.githubusercontent.com/stanfordnlp/contract-nli/eced6528dd3c1d14d73f9a87df8f7bdbc03126f9/resources/contract-nli.zip",
    "license": "CC-BY-4.0",
    "train_allowed": true,
    "natural": true
  },
  "qasper": {
    "kind": "huggingface",
    "repo": "allenai/qasper",
    "config": null,
    "loader": "pinned_parquet_export",
    "parquet_ref": "refs/convert/parquet",
    "parquet_config": "qasper",
    "expected_splits": [
      "train",
      "validation",
      "test"
    ],
    "license": "CC-BY-4.0",
    "train_allowed": true,
    "natural": true,
    "revision": "fdc9d8214fbab5dd782958601db4d678e6934a54",
    "data_revision": "06806e4608976fc2fac0a090ac425d5b2b29caf4",
    "parquet_revision": "06806e4608976fc2fac0a090ac425d5b2b29caf4"
  }
}


## 6. State-centric normalized schema

The corpus is stored as four linked Parquet tables so the state text exists once:

```text
states.parquet    state_id, component_id, source, upstream_split, text, hashes, provenance
      │
      └── questions.parquet   question_id, primitive, family, target, applicability reason
                ├── options.parquet    semantic option IDs and descriptions
                └── evidence.parquet   character spans and/or evidence text
```

`component_id` is the split boundary. Exact duplicate text, alternate annotations of the same upstream document, and declared upstream groups must never cross partitions.


In [25]:
from collections import defaultdict, Counter
import pandas as pd
import numpy as np

SPLITS = ("train", "development", "calibration_fit", "calibration_gate", "policy_development", "final")
NONE_REASONS = ("omitted_option", "out_of_scope", "insufficient_evidence", "unobservable_state")

def text_sha(text):
    return hashlib.sha256(text.encode("utf-8")).hexdigest()

def stable_id(*parts, prefix="od"):
    payload = "\x1f".join(map(str, parts)).encode()
    return f"{prefix}:{hashlib.sha256(payload).hexdigest()[:24]}"

def json_text(obj):
    return json.dumps(obj, sort_keys=True, ensure_ascii=False, separators=(",", ":"))

class MQCorpus:
    def __init__(self):
        self.states = {}
        self.questions = {}
        self.options = []
        self.evidence = []
        self.criteria = {}

    def add_state(self, *, state_id, source, upstream_split, text, upstream_group, metadata=None):
        digest = text_sha(text)
        component_id = stable_id(source, upstream_group or digest, digest, prefix="component")
        record = {
            "state_id": state_id,
            "component_id": component_id,
            "source": source,
            "upstream_split": upstream_split,
            "upstream_group": str(upstream_group),
            "text": text,
            "text_sha256": digest,
            "split": None,
            "eligible_for_training": bool(SOURCE_REGISTRY[source]["train_allowed"]),
            "metadata_json": json_text(metadata or {}),
        }
        prior = self.states.get(state_id)
        if prior is not None and prior != record:
            raise ValueError(f"Conflicting state: {state_id}")
        self.states[state_id] = record

    def add_question(self, *, question_id, state_id, primitive, family, rubric_family,
                     instruction, options, target_kind, target_option_id=None,
                     semantic_none_label=None, applicability_reason=None, metadata=None,
                     criteria_id=None):
        if state_id not in self.states:
            raise KeyError(state_id)
        if question_id in self.questions:
            raise ValueError(f"Duplicate question: {question_id}")
        if target_kind not in ("option", "semantic_none"):
            raise ValueError(target_kind)
        option_ids = [x["id"] for x in options]
        if len(option_ids) != len(set(option_ids)) or not option_ids:
            raise ValueError(f"Invalid options for {question_id}")
        if target_kind == "option" and target_option_id not in option_ids:
            raise ValueError(f"Target absent from options: {question_id}")
        if target_kind == "semantic_none" and applicability_reason not in NONE_REASONS:
            raise ValueError(f"Missing semantic-none reason: {question_id}")
        self.questions[question_id] = {
            "question_id": question_id,
            "state_id": state_id,
            "primitive": primitive,
            "family": family,
            "rubric_family": rubric_family,
            "criteria_id": criteria_id or rubric_family,
            "instruction": instruction,
            "target_kind": target_kind,
            "target_option_id": target_option_id,
            "semantic_none_label": semantic_none_label,
            "applicability_reason": applicability_reason,
            "metadata_json": json_text(metadata or {}),
        }
        for position, opt in enumerate(options):
            self.options.append({
                "question_id": question_id,
                "option_id": opt["id"],
                "position": position,
                "description": opt["description"],
                "value": opt.get("value"),
            })

    def add_evidence(self, question_id, *, char_start=None, char_end=None, text=None, evidence_type="support"):
        if question_id not in self.questions:
            raise KeyError(question_id)
        self.evidence.append({
            "question_id": question_id,
            "evidence_id": stable_id(question_id, len(self.evidence), char_start, char_end, text, prefix="evidence"),
            "char_start": char_start,
            "char_end": char_end,
            "text": text,
            "evidence_type": evidence_type,
        })

    def register_criterion(self, criteria_id, description, source, metadata=None):
        record = {"criteria_id": criteria_id, "description": description, "source": source,
                  "metadata_json": json_text(metadata or {})}
        prior = self.criteria.get(criteria_id)
        if prior is not None and prior != record:
            raise ValueError(f"Criterion drift: {criteria_id}")
        self.criteria[criteria_id] = record

    def frames(self):
        return {
            "states": pd.DataFrame(self.states.values()),
            "questions": pd.DataFrame(self.questions.values()),
            "options": pd.DataFrame(self.options),
            "evidence": pd.DataFrame(self.evidence),
            "criteria": pd.DataFrame(self.criteria.values()),
        }


## 7. ContractNLI adapter — the core state-once source

ContractNLI contributes one NDA state with 17 fixed hypotheses, three source outcomes, and evidence spans. The official Stanford release is pinned by repository commit and downloaded from that commit rather than from a mirror with conflicting license metadata.


In [26]:
import urllib.request

def download_file(url, destination):
    destination.parent.mkdir(parents=True, exist_ok=True)
    if destination.is_file():
        return destination
    tmp = destination.with_suffix(destination.suffix + ".partial")
    urllib.request.urlretrieve(url, tmp)
    os.replace(tmp, destination)
    return destination

def infer_upstream_split(path):
    name = path.name.lower()
    if "train" in name:
        return "train"
    if "dev" in name or "valid" in name:
        return "validation"
    if "test" in name:
        return "test"
    return "unknown"

def add_contractnli(corpus, limit_states=None):
    spec = RESOLVED_SOURCES["contractnli"]
    raw = LOCAL_RUN / "raw" / "contractnli.zip"
    unpacked = LOCAL_RUN / "raw" / "contractnli"
    download_file(spec["url"], raw)
    if not unpacked.exists():
        unpacked.mkdir(parents=True)
        safe_extract(raw, unpacked)

    dataset_files = []
    for path in sorted(unpacked.rglob("*.json")):
        try:
            obj = json.loads(path.read_text())
        except Exception:
            continue
        if isinstance(obj, dict) and "documents" in obj and "labels" in obj:
            dataset_files.append((path, obj))
    if not dataset_files:
        raise RuntimeError("No ContractNLI dataset JSON found in the official archive.")

    per_file_limit = None if limit_states is None else max(1, int(np.ceil(limit_states / len(dataset_files))))
    for path, payload in dataset_files:
        upstream_split = infer_upstream_split(path)
        labels = payload["labels"]
        for criterion_id, label in labels.items():
            corpus.register_criterion(
                f"contractnli:{criterion_id}", label["hypothesis"], "contractnli",
                {"short_description": label.get("short_description")},
            )
        file_states = 0
        for doc in payload["documents"]:
            if per_file_limit is not None and file_states >= per_file_limit:
                break
            state_id = f"contractnli:{upstream_split}:{doc['id']}"
            corpus.add_state(
                state_id=state_id,
                source="contractnli",
                upstream_split=upstream_split,
                text=doc["text"],
                upstream_group=doc["id"],
                metadata={
                    "file_name": doc.get("file_name"),
                    "document_type": doc.get("document_type"),
                    "url": doc.get("url"),
                    "source_revision": spec["revision"],
                },
            )
            annotations = doc["annotation_sets"][0]["annotations"]
            for criterion_id, annotation in annotations.items():
                choice = annotation["choice"]
                mapping = {"Entailment": "entailed", "Contradiction": "contradicted"}
                qid = f"{state_id}:{criterion_id}"
                is_none = choice == "NotMentioned"
                corpus.add_question(
                    question_id=qid,
                    state_id=state_id,
                    primitive="choice",
                    family="contract_entailment",
                    rubric_family=f"contractnli:{criterion_id}",
                    criteria_id=f"contractnli:{criterion_id}",
                    instruction=labels[criterion_id]["hypothesis"],
                    options=[
                        {"id": "entailed", "description": "The contract supports the statement."},
                        {"id": "contradicted", "description": "The contract conflicts with the statement."},
                    ],
                    target_kind="semantic_none" if is_none else "option",
                    target_option_id=None if is_none else mapping[choice],
                    semantic_none_label="not_mentioned" if is_none else None,
                    applicability_reason="insufficient_evidence" if is_none else None,
                    metadata={"source_label": choice, "source_file": path.name},
                )
                for span_index in annotation.get("spans", []):
                    start, end = doc["spans"][span_index]
                    corpus.add_evidence(qid, char_start=start, char_end=end,
                                        text=doc["text"][start:end], evidence_type="support")
            file_states += 1


## 8. Natural-document and auxiliary adapters

QASPER is normalized as an **answerability/evidence** task, not as open-ended answer generation. MultiRC keeps each candidate's supported/not-supported semantics explicit. SQuAD 2.0 contributes answerability/insufficient-evidence supervision. MASSIVE contributes dynamic candidate descriptions but is not treated as multi-question evidence.


In [27]:
from collections import defaultdict
from datasets import load_dataset
from huggingface_hub import HfApi, hf_hub_url

def load_pinned_parquet_export(name, spec):
    token = os.environ.get("HF_TOKEN")
    revision = spec["parquet_revision"]
    prefix = spec["parquet_config"].strip("/") + "/"
    files = HfApi(token=token).list_repo_files(
        spec["repo"], repo_type="dataset", revision=revision
    )
    data_files = defaultdict(list)
    for path in files:
        if not path.startswith(prefix) or not path.endswith(".parquet"):
            continue
        relative = path[len(prefix):]
        if "/" not in relative:
            continue
        split = relative.split("/", 1)[0]
        data_files[split].append(
            hf_hub_url(
                spec["repo"], path, repo_type="dataset", revision=revision
            )
        )

    expected = set(spec.get("expected_splits", ()))
    missing = expected.difference(data_files)
    if missing:
        raise RuntimeError(
            f"{name} pinned Parquet export is missing splits: {sorted(missing)}"
        )
    if not data_files:
        raise RuntimeError(f"No Parquet files found for {name} at {revision}")

    ordered = {split: sorted(paths) for split, paths in sorted(data_files.items())}
    print(
        f"Loading {name} from pinned Parquet revision {revision[:12]} "
        f"with splits {list(ordered)}"
    )
    return load_dataset("parquet", data_files=ordered, token=token)

def hf_dataset(name):
    spec = RESOLVED_SOURCES[name]
    if spec.get("loader") == "pinned_parquet_export":
        return load_pinned_parquet_export(name, spec)
    kwargs = {"revision": spec["revision"], "token": os.environ.get("HF_TOKEN")}
    if spec.get("config"):
        return load_dataset(spec["repo"], spec["config"], **kwargs)
    return load_dataset(spec["repo"], **kwargs)

def flatten_qasper_state(row):
    parts = [row.get("title", ""), row.get("abstract", "")]
    full = row.get("full_text") or {}
    sections = full.get("section_name", [])
    paragraphs = full.get("paragraphs", [])
    for section, paras in zip(sections, paragraphs):
        if section:
            parts.append(f"\n## {section}")
        if isinstance(paras, str):
            parts.append(paras)
        else:
            parts.extend(str(p) for p in paras)
    return "\n".join(p for p in parts if p)

def list_at(mapping, key, index, default=None):
    value = (mapping or {}).get(key, default)
    if isinstance(value, list) and index < len(value):
        return value[index]
    return default

def qasper_answer_records(qas, index):
    raw = list_at(qas, "answers", index, [])
    if isinstance(raw, dict):
        raw = raw.get("answer", raw.get("answers", []))
    if isinstance(raw, list) and len(raw) == 1 and isinstance(raw[0], dict) and "answer" in raw[0]:
        raw = raw[0]["answer"]
    return raw if isinstance(raw, list) else []

def add_qasper(corpus, limit_states=None):
    ds = hf_dataset("qasper")
    per_split_limit = None if limit_states is None else max(1, int(np.ceil(limit_states / len(ds))))
    for upstream_split, split_ds in ds.items():
        split_count = 0
        for row in split_ds:
            if per_split_limit is not None and split_count >= per_split_limit:
                break
            paper_id = str(row.get("id", row.get("paper_id", split_count)))
            state_id = f"qasper:{upstream_split}:{paper_id}"
            state_text = flatten_qasper_state(row)
            corpus.add_state(
                state_id=state_id, source="qasper", upstream_split=upstream_split,
                text=state_text, upstream_group=paper_id,
                metadata={
                    "title": row.get("title"),
                    "source_revision": RESOLVED_SOURCES["qasper"]["data_revision"],
                    "source_repo_revision": RESOLVED_SOURCES["qasper"]["revision"],
                },
            )
            qas = row.get("qas") or {}
            questions = qas.get("question", [])
            for i, question in enumerate(questions):
                source_qid = str(list_at(qas, "question_id", i, i))
                answers = qasper_answer_records(qas, i)
                flags = [bool(a.get("unanswerable", False)) for a in answers if isinstance(a, dict)]
                unanswerable = bool(flags) and sum(flags) >= (len(flags) + 1) // 2
                qid = f"{state_id}:{source_qid}"
                corpus.add_question(
                    question_id=qid, state_id=state_id, primitive="choice",
                    family="paper_answerability", rubric_family="qasper:answerability",
                    criteria_id="qasper:answerability",
                    instruction=str(question),
                    options=[{"id": "answerable", "description": "The paper contains enough evidence to answer the question."}],
                    target_kind="semantic_none" if unanswerable else "option",
                    target_option_id=None if unanswerable else "answerable",
                    semantic_none_label="unanswerable" if unanswerable else None,
                    applicability_reason="insufficient_evidence" if unanswerable else None,
                    metadata={"source_question_id": source_qid, "annotator_answers": len(answers)},
                )
                evidence_strings = []
                for answer in answers:
                    if not isinstance(answer, dict):
                        continue
                    evidence_strings.extend(answer.get("evidence") or [])
                for evidence_text in sorted(set(map(str, evidence_strings))):
                    start = state_text.find(evidence_text)
                    corpus.add_evidence(
                        qid,
                        char_start=start if start >= 0 else None,
                        char_end=(start + len(evidence_text)) if start >= 0 else None,
                        text=evidence_text,
                        evidence_type="support",
                    )
            corpus.register_criterion(
                "qasper:answerability",
                "Determine whether the paper contains enough evidence to answer the question.",
                "qasper",
            )
            split_count += 1

def add_multirc(corpus, limit_states=None):
    ds = hf_dataset("multirc")
    states_seen = set()
    for upstream_split, split_ds in ds.items():
        for row in split_ds:
            idx = row.get("idx") or {}
            paragraph_id = str(idx.get("paragraph", text_sha(row["paragraph"])[:16]))
            state_id = f"multirc:{upstream_split}:{paragraph_id}"
            if state_id not in states_seen:
                if limit_states is not None and len(states_seen) >= limit_states:
                    return
                corpus.add_state(
                    state_id=state_id, source="multirc", upstream_split=upstream_split,
                    text=row["paragraph"], upstream_group=paragraph_id,
                    metadata={"source_revision": RESOLVED_SOURCES["multirc"]["revision"]},
                )
                states_seen.add(state_id)
            if int(row.get("label", -1)) not in (0, 1):
                continue
            qid = f"{state_id}:q{idx.get('question', 0)}:a{idx.get('answer', 0)}"
            corpus.add_question(
                question_id=qid, state_id=state_id, primitive="noul",
                family="candidate_support", rubric_family="multirc:support",
                criteria_id="multirc:support",
                instruction=f"Question: {row['question']}\nCandidate answer: {row['answer']}\nIs the candidate supported?",
                options=[
                    {"id": "false", "description": "The candidate answer is not supported."},
                    {"id": "true", "description": "The candidate answer is supported."},
                ],
                target_kind="option", target_option_id="true" if int(row["label"]) else "false",
                metadata={"source_idx": idx},
            )
    corpus.register_criterion("multirc:support", "Judge each candidate answer independently.", "multirc")

def add_squad2(corpus, limit_states=None):
    ds = hf_dataset("squad2")
    states_seen = set()
    for upstream_split, split_ds in ds.items():
        for row in split_ds:
            context_hash = text_sha(row["context"])[:20]
            state_id = f"squad2:{upstream_split}:{context_hash}"
            if state_id not in states_seen:
                if limit_states is not None and len(states_seen) >= limit_states:
                    return
                corpus.add_state(
                    state_id=state_id, source="squad2", upstream_split=upstream_split,
                    text=row["context"], upstream_group=context_hash,
                    metadata={"title": row.get("title"), "source_revision": RESOLVED_SOURCES["squad2"]["revision"]},
                )
                states_seen.add(state_id)
            answers = row.get("answers") or {}
            answer_texts = answers.get("text") or []
            starts = answers.get("answer_start") or []
            answerable = len(answer_texts) > 0
            qid = f"{state_id}:{row['id']}"
            corpus.add_question(
                question_id=qid, state_id=state_id, primitive="choice",
                family="passage_answerability", rubric_family="squad2:answerability",
                criteria_id="squad2:answerability", instruction=row["question"],
                options=[{"id": "answerable", "description": "The passage contains enough evidence to answer the question."}],
                target_kind="option" if answerable else "semantic_none",
                target_option_id="answerable" if answerable else None,
                semantic_none_label=None if answerable else "unanswerable",
                applicability_reason=None if answerable else "insufficient_evidence",
            )
            for start, answer in zip(starts, answer_texts):
                corpus.add_evidence(qid, char_start=int(start), char_end=int(start) + len(answer), text=answer)
    corpus.register_criterion("squad2:answerability", "Determine whether the passage can answer the question.", "squad2")

def add_massive(corpus, limit_states=None, k=8):
    ds = hf_dataset("massive")
    count = 0
    for upstream_split, split_ds in ds.items():
        intent_feature = split_ds.features["intent"]
        names = list(intent_feature.names)
        for row in split_ds:
            if limit_states is not None and count >= limit_states:
                return
            utterance = row.get("utt", row.get("text"))
            label = int(row["intent"])
            digest = int(hashlib.sha256(str(row.get("id", count)).encode()).hexdigest()[:8], 16)
            distractors = [i for i in range(len(names)) if i != label]
            rng = random.Random(SEED + digest)
            rng.shuffle(distractors)
            chosen = [label] + distractors[:max(1, k - 1)]
            rng.shuffle(chosen)
            state_id = f"massive:{upstream_split}:{row.get('id', count)}"
            corpus.add_state(
                state_id=state_id, source="massive", upstream_split=upstream_split,
                text=utterance, upstream_group=row.get("id", count),
                metadata={"scenario": row.get("scenario"), "locale": row.get("locale"),
                          "source_revision": RESOLVED_SOURCES["massive"]["revision"]},
            )
            options = [{"id": names[i], "description": names[i].replace("_", " ")} for i in chosen]
            corpus.add_question(
                question_id=f"{state_id}:intent", state_id=state_id, primitive="choice",
                family="dynamic_intent", rubric_family="massive:intent",
                criteria_id="massive:intent", instruction="Which intent best describes this request?",
                options=options, target_kind="option", target_option_id=names[label],
                metadata={"full_intent_inventory_size": len(names)},
            )
            count += 1
    corpus.register_criterion("massive:intent", "Select an intent using its natural-language description.", "massive")


## 9. Build the corpus, assign group-safe splits, and audit

Upstream test partitions are never reassigned to training. Within each upstream partition, assignment is deterministic from `component_id`, source, and the declared split salt. This is output-blind and stable across row order.


In [28]:
def fraction(key):
    raw = hashlib.sha256(f"phase4a-v1|{key}".encode()).digest()[:8]
    return int.from_bytes(raw, "big") / 2**64

def assign_split(upstream_split, component_id, train_allowed=True):
    u = str(upstream_split).lower()
    r = fraction(component_id)
    if not train_allowed:
        return "calibration_gate" if r < 0.5 else "final"
    if u in ("test",):
        return "calibration_gate" if r < 0.5 else "final"
    if u in ("validation", "valid", "dev"):
        if r < 0.50: return "development"
        if r < 0.75: return "calibration_fit"
        return "policy_development"
    if r < 0.82: return "train"
    if r < 0.91: return "development"
    if r < 0.96: return "calibration_fit"
    return "policy_development"

corpus = MQCorpus()
limit = CFG["smoke_states_per_source"] if CFG["run_mode"] == "smoke" else None
for source in CFG["enabled_sources"]:
    if source in ("quality", "summeval"):
        raise RuntimeError(f"{source} is registered but its reviewed adapter is intentionally not implemented yet.")
    globals()[f"add_{source}"](corpus, limit_states=limit)

for state in corpus.states.values():
    state["split"] = assign_split(
        state["upstream_split"], state["component_id"], state["eligible_for_training"]
    )

frames = corpus.frames()
state_split = frames["states"].set_index("state_id")["split"].to_dict()
frames["questions"]["split"] = frames["questions"]["state_id"].map(state_split)

def audit_corpus(frames, smoke=False):
    errors, warnings = [], []
    states, questions, options, evidence = (frames[x] for x in ("states", "questions", "options", "evidence"))
    if states["state_id"].duplicated().any(): errors.append("duplicate state IDs")
    if questions["question_id"].duplicated().any(): errors.append("duplicate question IDs")
    if not set(questions["state_id"]).issubset(set(states["state_id"])): errors.append("orphan questions")
    if not set(options["question_id"]).issubset(set(questions["question_id"])): errors.append("orphan options")
    if len(evidence) and not set(evidence["question_id"]).issubset(set(questions["question_id"])): errors.append("orphan evidence")
    crossing = states.groupby("component_id")["split"].nunique()
    if (crossing > 1).any(): errors.append("connected/source component crosses splits")
    text_crossing = states.groupby("text_sha256")["split"].nunique()
    if (text_crossing > 1).any(): errors.append("exact duplicate state text crosses splits")
    option_sets = options.groupby("question_id")["option_id"].apply(set).to_dict()
    for row in questions.itertuples(index=False):
        if row.target_kind == "option" and row.target_option_id not in option_sets[row.question_id]:
            errors.append(f"target not offered: {row.question_id}")
        if row.target_kind == "semantic_none" and row.applicability_reason not in NONE_REASONS:
            errors.append(f"semantic none lacks reason: {row.question_id}")
    counts = states.groupby(["source", "split"]).size().unstack(fill_value=0)
    for source in states["source"].unique():
        missing = [s for s in SPLITS if counts.loc[source].get(s, 0) == 0]
        if missing:
            (warnings if smoke else errors).append(f"{source} missing state splits: {missing}")
    return errors, warnings

errors, warnings = audit_corpus(frames, smoke=CFG["run_mode"] == "smoke")
if errors:
    raise RuntimeError("Corpus audit failed:\n- " + "\n- ".join(errors))
if warnings:
    print("SMOKE WARNINGS:\n- " + "\n- ".join(warnings))

display(frames["states"].groupby(["source", "split"]).size().rename("states").reset_index())
question_audit = frames["questions"].merge(
    frames["states"][["state_id", "source"]], on="state_id", validate="many_to_one"
)
display(question_audit.groupby(["source", "split"]).size().rename("questions").reset_index())
print("States:", len(frames["states"]), "Questions:", len(frames["questions"]), "Options:", len(frames["options"]))


Loading qasper from pinned Parquet revision 06806e460897 with splits ['test', 'train', 'validation']


,source,split,states
0,contractnli,calibration_fit,34
1,contractnli,calibration_gate,71
2,contractnli,development,70
3,contractnli,final,52
4,contractnli,policy_development,39
5,contractnli,train,341
6,qasper,calibration_fit,119
7,qasper,calibration_gate,199
8,qasper,development,242
9,qasper,final,217


,source,split,questions
0,contractnli,calibration_fit,578
1,contractnli,calibration_gate,1207
2,contractnli,development,1190
3,contractnli,final,884
4,contractnli,policy_development,663
5,contractnli,train,5797
6,qasper,calibration_fit,377
7,qasper,calibration_gate,692
8,qasper,development,836
9,qasper,final,759


States: 2192 Questions: 15368 Options: 25687


## 10. Persist and lock the normalized corpus

The lock covers canonical table hashes, source revisions, criteria definitions, split counts, and the declared study contract. Changing a label, criterion, split, or source revision requires a new run rather than an in-place edit.


In [29]:
CORPUS_LOCAL = LOCAL_RUN / "opendecision-mq-v1"
CORPUS_DRIVE = DRIVE_RUN / "opendecision-mq-v1"
for root in (CORPUS_LOCAL, CORPUS_DRIVE):
    root.mkdir(parents=True, exist_ok=True)

def stable_frame(df, sort_cols):
    return df.sort_values(sort_cols).reset_index(drop=True)

table_sorts = {
    "states": ["state_id"],
    "questions": ["question_id"],
    "options": ["question_id", "position"],
    "evidence": ["question_id", "evidence_id"],
    "criteria": ["criteria_id"],
}
TABLE_HASHES = {}
for name, df in frames.items():
    df = stable_frame(df, table_sorts[name])
    frames[name] = df
    local_path = CORPUS_LOCAL / f"{name}.parquet"
    drive_path = CORPUS_DRIVE / f"{name}.parquet"
    df.to_parquet(local_path, index=False)
    shutil.copy2(local_path, drive_path)
    TABLE_HASHES[name] = sha256_file(local_path)

CORPUS_MANIFEST = {
    "schema": "opendecision-mq-manifest/v1",
    "run_id": RUN_ID,
    "declared_contract_sha256": DECLARED_CONTRACT["sha256"],
    "sources": RESOLVED_SOURCES,
    "tables": TABLE_HASHES,
    "counts": {name: len(df) for name, df in frames.items()},
    "state_counts_by_source_split": (
        frames["states"].groupby(["source", "split"]).size()
        .rename("count").reset_index().to_dict("records")
    ),
    "question_counts_by_source_family": (
        frames["questions"].merge(frames["states"][["state_id", "source"]], on="state_id")
        .groupby(["source", "family"]).size().rename("count").reset_index().to_dict("records")
    ),
    "final_labels_or_predictions_opened": False,
}
CORPUS_LOCK = {
    "schema": "opendecision-corpus-lock/v1",
    "manifest_sha256": sha256_obj(CORPUS_MANIFEST),
    "tables": TABLE_HASHES,
    "locked_before_model_predictions": True,
}
for root in (CORPUS_LOCAL, CORPUS_DRIVE):
    (root / "manifest.json").write_text(json.dumps(CORPUS_MANIFEST, indent=2))
    (root / "corpus_lock.json").write_text(json.dumps(CORPUS_LOCK, indent=2))
    (root / "criteria_registry.json").write_text(
        json.dumps(frames["criteria"].to_dict("records"), indent=2, ensure_ascii=False)
    )
print(json.dumps(CORPUS_LOCK, indent=2))


{
  "schema": "opendecision-corpus-lock/v1",
  "manifest_sha256": "1aad04ca7d37ea8be640a51878857c6d2c501d635e913eca12234cb548f8399d",
  "tables": {
    "states": "7e6333202ad934e66148724a565b27176fcaad4133a566d2c8f8b57044ef3120",
    "questions": "549b446f609a9407eceb5df7ef4c0f14779bc73d72396ae749424626d8777434",
    "options": "cbd40c9162909c6c637296e9c8f408e0ba1bece0aca1786cef98f73d242f6cf2",
    "evidence": "566e0e69e8f57ed0ce626021647a3b770a89ae5a3eb318e1d4f8bea284f2c6be",
    "criteria": "1e3e62c93506fc7add11c88dc7232dba9f6c0b9ee8db2ee4b465fb283a91430b"
  },
  "locked_before_model_predictions": true
}


## 11. Predeclared robustness views

Variants are generated only after the base corpus is locked. They retain `base_question_id` and stay in the same split. Option stability is compared after aligning by semantic option ID.

Implemented initial views:

- reverse and deterministically rotate displayed option order;
- meaning-preserving criterion wrapper;
- append owned irrelevant context;
- remove annotated evidence text where an exact span is available;
- rename opaque machine IDs without changing semantic descriptions.

These are diagnostics, not automatically added to training.


In [30]:
OWNED_IRRELEVANT_CONTEXT = (
    "Administrative note: this paragraph is unrelated to the decision criterion and "
    "contains no evidence for any offered option."
)

def robustness_views(state, question, options, evidence_rows):
    views = []
    reversed_options = list(reversed(options))
    views.append({"kind": "option_reverse", "state": state, "question": question,
                  "options": reversed_options})
    wrapped = dict(question)
    wrapped["instruction"] = "Based only on the supplied state, assess this criterion: " + question["instruction"]
    views.append({"kind": "criterion_wrapper", "state": state, "question": wrapped,
                  "options": options})
    views.append({"kind": "irrelevant_append", "state": state + "\n\n" + OWNED_IRRELEVANT_CONTEXT,
                  "question": question, "options": options})
    deleted = state
    for row in evidence_rows:
        if row.get("text"):
            deleted = deleted.replace(row["text"], " [EVIDENCE REMOVED] ")
    if deleted != state:
        views.append({"kind": "evidence_deleted", "state": deleted, "question": question,
                      "options": options})
    renamed = [dict(opt, option_id=f"opaque_{i:03d}") for i, opt in enumerate(options)]
    views.append({"kind": "opaque_id_rename", "state": state, "question": question,
                  "options": renamed})
    return views

def align_probabilities(rows, id_key="option_id", p_key="probability"):
    return {r[id_key]: float(r[p_key]) for r in rows}


## 12. GPU Stage A — frozen reference predictions and feature caches

This stage loads the exact selected 4B runtime once. It:

1. creates the current historical candidate-conditioned reference predictions on non-final partitions;
2. truncates each state once using a declared head/tail policy;
3. caches the state-only final hidden sequence and tokenizer offsets;
4. exports the frozen Qwen input-embedding matrix for cheap question/candidate token lookup;
5. caches **candidate-conditioned 4B features** for a matched B0 control trained on the same new labels;
6. closes and unloads the 4B runtime before readout/query-model optimization.

The baseline and StateQuery path receive the same effective state text. Model loading and cache writes are outside benchmark timing.

**Attribution rule:** B1/B2 must be compared primarily with **B0**, not only with the old Phase 2IJ head. Otherwise new supervision and a new architecture are confounded. The historical selected head remains a regression/reference line.


In [31]:
GPU_STAGE_READY = False
if not CFG["execute_gpu"]:
    print("GPU stage skipped. Set CFG['execute_gpu']=True and rerun from this cell.")
else:
    import torch
    from safetensors.torch import save_file

    if not torch_version.startswith("2.11."):
        raise RuntimeError("Torch 2.11.x is required for the selected reference runtime.")
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA GPU required; no accidental CPU 4B fallback.")

    import copy
    sys.path.insert(0, str(LOCAL_BUNDLE / "implementation"))
    from odij_bundle import BundlePredictor
    from odij_learning import softmax as reference_softmax

    reference = BundlePredictor(LOCAL_BUNDLE, LOCAL_RUN / "reference_runtime", allow_cpu_test=False)
    rt = reference.predictor.rt
    assert type(rt.model).__name__ == "Qwen3_5TextModel"
    assert rt.hidden == MODEL_CONTRACT["hidden_size"]
    assert sum(p.numel() for p in rt.model.parameters()) == MODEL_CONTRACT["parameter_count"]

    tokenizer = rt.tokenizer
    device = next(rt.model.parameters()).device
    FEATURE_LOCAL = LOCAL_RUN / "state_features"
    FEATURE_DRIVE = DRIVE_RUN / "state_features"
    for root in (FEATURE_LOCAL, FEATURE_DRIVE): root.mkdir(exist_ok=True)

    def effective_state_text(text, max_tokens):
        ids = tokenizer(text, add_special_tokens=False)["input_ids"]
        if len(ids) <= max_tokens:
            return text, False
        head = int(max_tokens * 0.75)
        tail = max_tokens - head - 16
        rendered = (
            tokenizer.decode(ids[:head], skip_special_tokens=True)
            + "\n[...TRUNCATED BY PHASE4A CONTRACT...]\n"
            + tokenizer.decode(ids[-tail:], skip_special_tokens=True)
        )
        # The marker can tokenize longer than its reserved budget.
        final_ids = tokenizer(rendered, add_special_tokens=False)["input_ids"][:max_tokens]
        return tokenizer.decode(final_ids, skip_special_tokens=True), True

    def atomic_safetensors(tensors, destination):
        tmp = destination.with_suffix(destination.suffix + ".partial")
        save_file(tensors, tmp)
        os.replace(tmp, destination)

    state_rows = frames["states"].to_dict("records")
    state_index = []
    control_index = []
    reference_rows = []
    question_lookup = frames["questions"].set_index("question_id").to_dict("index")
    options_grouped = {
        qid: group.sort_values("position").to_dict("records")
        for qid, group in frames["options"].groupby("question_id")
    }

    for n, state in enumerate(state_rows, 1):
        effective, truncated = effective_state_text(state["text"], CFG["max_state_tokens"])
        encoded = tokenizer(
            effective, add_special_tokens=False, return_tensors="pt",
            return_offsets_mapping=True, truncation=True, max_length=CFG["max_state_tokens"],
        )
        offsets = encoded.pop("offset_mapping")[0]
        model_inputs = {k: v.to(device) for k, v in encoded.items()}
        with torch.inference_mode():
            hidden = rt.model(**model_inputs, use_cache=False, return_dict=True).last_hidden_state[0]
        cache_dtype = getattr(torch, CFG["feature_cache_dtype"])
        hidden_cpu = hidden.detach().to(device="cpu", dtype=cache_dtype).contiguous()
        key = hashlib.sha256(state["state_id"].encode()).hexdigest()[:32]
        local_path = FEATURE_LOCAL / f"{key}.safetensors"
        atomic_safetensors({
            "hidden": hidden_cpu,
            "input_ids": encoded["input_ids"][0].to(torch.int32).cpu(),
            "offsets": offsets.to(torch.int32).cpu(),
        }, local_path)
        shutil.copy2(local_path, FEATURE_DRIVE / local_path.name)
        state_index.append({
            "state_id": state["state_id"], "file": local_path.name,
            "sha256": sha256_file(local_path), "tokens": int(hidden_cpu.shape[0]),
            "hidden": int(hidden_cpu.shape[1]), "truncated": truncated,
            "effective_state_text": effective,
        })

        if state["split"] != "final":
            control_tensors = {}
            state_questions = frames["questions"][frames["questions"]["state_id"] == state["state_id"]]
            for q in state_questions.to_dict("records"):
                option_rows = options_grouped[q["question_id"]]
                # Use the exact selected predictor directly because the old public
                # request validator knows only the Phase 2IJ family registry. This
                # changes no renderer, feature, head, calibration, or probability math.
                predictor_q = {
                    "id": q["question_id"], "family": q["family"],
                    "rubric_family": q["rubric_family"], "primitive": q["primitive"],
                    "instruction": q["instruction"],
                    "options": [
                        {"id": o["option_id"], "label": o["description"],
                         "criteria": o["description"]}
                        for o in option_rows
                    ],
                }
                predictor_row = {
                    "id": q["question_id"], "group": state["component_id"],
                    "state_id": state["state_id"], "state": effective,
                    "split": state["split"], "source_kind": "natural",
                    "q": predictor_q, "target": 0,
                }
                conditioned = np.asarray(
                    rt.row_features(predictor_row, bundle_profile["layout"], memo=False),
                    dtype=np.float32,
                )
                with torch.no_grad():
                    logits = reference.predictor.d(
                        torch.from_numpy(conditioned), q["primitive"] == "choice"
                    ).detach().cpu().numpy()
                probabilities = np.asarray(
                    reference_softmax(logits, bundle_profile["calibration"]["selected"])
                )
                control_key = "q_" + hashlib.sha256(q["question_id"].encode()).hexdigest()[:24]
                control_tensors[control_key] = torch.from_numpy(conditioned).to(cache_dtype)
                control_index.append({
                    "question_id": q["question_id"], "state_id": state["state_id"],
                    "file": f"control_{key}.safetensors", "key": control_key,
                    "candidates": int(conditioned.shape[0]), "hidden": int(conditioned.shape[1]),
                })
                option_probs = {
                    o["option_id"]: float(probabilities[i]) for i, o in enumerate(option_rows)
                }
                none_probability = float(probabilities[-1]) if q["primitive"] == "choice" else None
                selected_index = int(probabilities.argmax())
                selected_id = (
                    None if q["primitive"] == "choice" and selected_index == len(option_rows)
                    else option_rows[selected_index]["option_id"]
                )
                reference_rows.append({
                    "question_id": q["question_id"], "selected_id": selected_id,
                    "option_probabilities_json": json_text(option_probs),
                    "none_probability": none_probability,
                })
            control_local = FEATURE_LOCAL / f"control_{key}.safetensors"
            atomic_safetensors(control_tensors, control_local)
            shutil.copy2(control_local, FEATURE_DRIVE / control_local.name)
            control_sha = sha256_file(control_local)
            for record in control_index:
                if record["file"] == control_local.name:
                    record["file_sha256"] = control_sha
        if n % 10 == 0:
            print(f"Cached {n}/{len(state_rows)} states")

    embedding = rt.model.get_input_embeddings().weight.detach().to("cpu", dtype=torch.bfloat16).contiguous()
    emb_local = FEATURE_LOCAL / "qwen_input_embedding.bf16.safetensors"
    atomic_safetensors({"weight": embedding}, emb_local)
    shutil.copy2(emb_local, FEATURE_DRIVE / emb_local.name)

    FEATURE_MANIFEST = {
        "schema": "opendecision-state-feature-cache/v1",
        "run_id": RUN_ID,
        "corpus_manifest_sha256": CORPUS_LOCK["manifest_sha256"],
        "model": MODEL_CONTRACT,
        "dtype": CFG["feature_cache_dtype"],
        "max_state_tokens": CFG["max_state_tokens"],
        "state_files": state_index,
        "candidate_conditioned_control_files": control_index,
        "embedding_file": emb_local.name,
        "embedding_sha256": sha256_file(emb_local),
        "reference_predictions_exclude_final": True,
    }
    reference_df = pd.DataFrame(reference_rows)
    reference_df.to_parquet(LOCAL_RUN / "reference_nonfinal_predictions.parquet", index=False)
    shutil.copy2(LOCAL_RUN / "reference_nonfinal_predictions.parquet",
                 DRIVE_RUN / "reference_nonfinal_predictions.parquet")
    for root in (FEATURE_LOCAL, FEATURE_DRIVE):
        (root / "manifest.json").write_text(json.dumps(FEATURE_MANIFEST, indent=2))

    reference.close()
    del reference, rt, embedding, hidden, hidden_cpu
    gc.collect(); torch.cuda.empty_cache()
    GPU_STAGE_READY = True
    print("Feature cache complete; 4B runtime unloaded.")


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/426 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Cached 10/2192 states
Cached 20/2192 states
Cached 30/2192 states
Cached 40/2192 states
Cached 50/2192 states
Cached 60/2192 states
Cached 70/2192 states
Cached 80/2192 states
Cached 90/2192 states
Cached 100/2192 states
Cached 110/2192 states
Cached 120/2192 states
Cached 130/2192 states
Cached 140/2192 states
Cached 150/2192 states
Cached 160/2192 states
Cached 170/2192 states
Cached 180/2192 states
Cached 190/2192 states
Cached 200/2192 states
Cached 210/2192 states
Cached 220/2192 states
Cached 230/2192 states
Cached 240/2192 states
Cached 250/2192 states
Cached 260/2192 states
Cached 270/2192 states
Cached 280/2192 states
Cached 290/2192 states
Cached 300/2192 states
Cached 310/2192 states
Cached 320/2192 states
Cached 330/2192 states
Cached 340/2192 states
Cached 350/2192 states
Cached 360/2192 states
Cached 370/2192 states
Cached 380/2192 states
Cached 390/2192 states
Cached 400/2192 states
Cached 410/2192 states
Cached 420/2192 states
Cached 430/2192 states
Cached 440/2192 stat

## 13. B0, B1, and B2 representation architectures

- **B0 matched control:** rerun frozen Qwen over every `state + question + candidate`, then train a new readout on those cached features under the same corpus/splits.
- **B1 direct:** a question query and every question+candidate query cross-attend directly to the projected state sequence.
- **B2 factorized:** 8 learned evidence slots cross-attend to the state once per question; candidate queries attend only to those compact slots.
- **B2R:** B2 plus a once-per-state bidirectional refiner after state projection.

The applicability head is permutation invariant over candidates. Candidate probabilities and semantic-none mass are assembled exactly as:

\[
P(c_i)=a\,\mathrm{softmax}(s)_i,\qquad P(\mathrm{none})=1-a
\]

No candidate softmax value is relabeled as operational confidence.


In [38]:
import torch
import torch.nn as nn
import torch.nn.functional as F

REASON_LABELS = ("applicable", "omitted_option", "out_of_scope", "insufficient_evidence", "unobservable_state")

class CrossBlock(nn.Module):
    def __init__(self, d_model, nhead, dropout=0.1, ff_mult=4):
        super().__init__()
        self.n1 = nn.LayerNorm(d_model)
        self.self_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.n2 = nn.LayerNorm(d_model)
        self.cross_attn = nn.MultiheadAttention(d_model, nhead, dropout=dropout, batch_first=True)
        self.n3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_mult * d_model), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(ff_mult * d_model, d_model), nn.Dropout(dropout),
        )

    def forward(self, x, memory, x_padding=None, memory_padding=None, return_attention=False):
        y = self.n1(x)
        y, _ = self.self_attn(y, y, y, key_padding_mask=x_padding, need_weights=False)
        x = x + y
        y = self.n2(x)
        y, weights = self.cross_attn(
            y, memory, memory,
            key_padding_mask=memory_padding,
            need_weights=return_attention,
            average_attn_weights=False,
        )
        x = x + y
        x = x + self.ff(self.n3(x))
        return x, weights

class QueryStack(nn.Module):
    def __init__(self, layers, d_model, nhead, dropout=0.1):
        super().__init__()
        self.layers = nn.ModuleList([CrossBlock(d_model, nhead, dropout) for _ in range(layers)])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x, memory, x_padding=None, memory_padding=None):
        attention = None
        for i, layer in enumerate(self.layers):
            x, attention = layer(
                x, memory, x_padding, memory_padding,
                return_attention=(i == len(self.layers) - 1),
            )
        return self.norm(x), attention

class StateRefiner(nn.Module):
    def __init__(self, layers, d_model, nhead):
        super().__init__()
        layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=4*d_model,
            dropout=0.1, activation="gelu", batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(layer, num_layers=layers) if layers else None

    def forward(self, memory, padding=None):
        return self.encoder(memory, src_key_padding_mask=padding) if self.encoder else memory

class StateQueryBase(nn.Module):
    def __init__(self, frozen_embedding, hidden_size=2560, d_model=768, heads=12,
                 layers=2, state_refiner_layers=0):
        super().__init__()
        self.token_embedding = nn.Embedding.from_pretrained(frozen_embedding, freeze=True)
        self.token_proj = nn.Linear(hidden_size, d_model, bias=False)
        self.state_proj = nn.Linear(hidden_size, d_model, bias=False)
        self.state_refiner = StateRefiner(state_refiner_layers, d_model, heads)
        self.rank_head = nn.Linear(d_model, 1)
        self.applicability = nn.Sequential(
            nn.Linear(3*d_model, d_model), nn.GELU(), nn.Linear(d_model, 1)
        )
        self.reason_head = nn.Sequential(
            nn.Linear(3*d_model, d_model), nn.GELU(), nn.Linear(d_model, len(REASON_LABELS))
        )
        self.d_model = d_model
        self.layers = layers
        self.heads = heads

    def embed(self, ids):
        return self.token_proj(self.token_embedding(ids))

    def project_state(self, hidden):
        memory = self.state_proj(hidden.float()).unsqueeze(0)
        return self.state_refiner(memory)

    def assemble(self, candidate_features, question_feature):
        feats = torch.stack(candidate_features, dim=0)
        pooled = torch.cat([question_feature, feats.mean(0), feats.max(0).values], dim=-1)
        candidate_logits = self.rank_head(feats).squeeze(-1)
        applicability_logit = self.applicability(pooled).squeeze(-1)
        reason_logits = self.reason_head(pooled)
        return candidate_logits, applicability_logit, reason_logits

class B0CandidateConditioned(nn.Module):
    """Matched-label control over frozen candidate-conditioned Qwen features."""
    def __init__(self, hidden_size=2560, d_model=768):
        super().__init__()
        self.feature_proj = nn.Sequential(
            nn.LayerNorm(hidden_size), nn.Linear(hidden_size, d_model), nn.GELU()
        )
        self.rank_head = nn.Linear(d_model, 1)
        self.applicability = nn.Sequential(
            nn.Linear(3*d_model, d_model), nn.GELU(), nn.Linear(d_model, 1)
        )
        self.reason_head = nn.Sequential(
            nn.Linear(3*d_model, d_model), nn.GELU(), nn.Linear(d_model, len(REASON_LABELS))
        )

    def forward_features(self, conditioned_features):
        feats = self.feature_proj(conditioned_features.float())
        mean = feats.mean(0); maximum = feats.max(0).values
        pooled = torch.cat([mean, mean, maximum], dim=-1)
        return {
            "candidate_logits": self.rank_head(feats).squeeze(-1),
            "applicability_logit": self.applicability(pooled).squeeze(-1),
            "reason_logits": self.reason_head(pooled),
            "evidence_attention": None,
            "question_memory": mean.unsqueeze(0),
        }

class B1StateQuery(StateQueryBase):
    def __init__(self, frozen_embedding, **kwargs):
        super().__init__(frozen_embedding, **kwargs)
        self.question_token = nn.Parameter(torch.randn(1, 1, self.d_model) * 0.02)
        self.candidate_token = nn.Parameter(torch.randn(1, 1, self.d_model) * 0.02)
        self.query = QueryStack(self.layers, self.d_model, self.heads)

    def query_once(self, learned, token_ids, memory):
        tokens = self.embed(token_ids).unsqueeze(0)
        x = torch.cat([learned, tokens], dim=1)
        out, attn = self.query(x, memory)
        return out[0, 0], attn

    def forward_memory(self, memory, question_ids, candidate_ids):
        qfeat, qattn = self.query_once(self.question_token, question_ids, memory)
        cfeats = []
        for ids in candidate_ids:
            joined = torch.cat([question_ids, ids])
            feat, _ = self.query_once(self.candidate_token, joined, memory)
            cfeats.append(feat)
        logits, app, reasons = self.assemble(cfeats, qfeat)
        return {"candidate_logits": logits, "applicability_logit": app,
                "reason_logits": reasons, "evidence_attention": qattn,
                "state_memory": memory, "question_memory": qfeat.unsqueeze(0)}

    def forward(self, state_hidden, question_ids, candidate_ids):
        return self.forward_memory(self.project_state(state_hidden), question_ids, candidate_ids)

class B2StateQuery(StateQueryBase):
    def __init__(self, frozen_embedding, question_latents=8, **kwargs):
        super().__init__(frozen_embedding, **kwargs)
        self.latents = nn.Parameter(torch.randn(1, question_latents, self.d_model) * 0.02)
        self.question_stack = QueryStack(self.layers, self.d_model, self.heads)
        self.candidate_token = nn.Parameter(torch.randn(1, 1, self.d_model) * 0.02)
        self.candidate_stack = QueryStack(max(1, self.layers // 2), self.d_model, self.heads)

    def forward_memory(self, memory, question_ids, candidate_ids):
        question_tokens = self.embed(question_ids).unsqueeze(0)
        slots = self.latents.expand(1, -1, -1)
        question_input = torch.cat([slots, question_tokens], dim=1)
        question_out, qattn = self.question_stack(question_input, memory)
        qmem = question_out[:, :self.latents.shape[1], :]
        qfeat = qmem.mean(dim=1)[0]
        cfeats = []
        for ids in candidate_ids:
            x = torch.cat([self.candidate_token, self.embed(ids).unsqueeze(0)], dim=1)
            out, _ = self.candidate_stack(x, qmem)
            cfeats.append(out[0, 0])
        logits, app, reasons = self.assemble(cfeats, qfeat)
        return {"candidate_logits": logits, "applicability_logit": app,
                "reason_logits": reasons, "evidence_attention": qattn,
                "state_memory": memory, "question_memory": qmem}

    def forward(self, state_hidden, question_ids, candidate_ids):
        return self.forward_memory(self.project_state(state_hidden), question_ids, candidate_ids)

def choice_distribution(candidate_logits, applicability_logit, rank_temperature=1.0, app_temperature=1.0):
    conditional = torch.softmax(candidate_logits / rank_temperature, dim=-1)
    applicability = torch.sigmoid(applicability_logit / app_temperature)
    probs = torch.cat([applicability * conditional, (1.0 - applicability).reshape(1)])
    torch.testing.assert_close(probs.sum(), torch.ones((), device=probs.device), atol=1e-6, rtol=0)
    return probs

# CPU shape/permutation contract: no model weights or corpus labels are used.
torch.manual_seed(SEED)
dummy_embedding = torch.randn(101, 32)
test_model = B1StateQuery(dummy_embedding, hidden_size=32, d_model=24, heads=4, layers=2)
test_model.eval()
state = torch.randn(11, 32)
qids = torch.tensor([1, 2, 3])
candidates = [torch.tensor([4, 5]), torch.tensor([6, 7, 8]), torch.tensor([9])]
with torch.no_grad():
    a = test_model(state, qids, candidates)
    b = test_model(state, qids, [candidates[2], candidates[0], candidates[1]])
    pa = choice_distribution(a["candidate_logits"], a["applicability_logit"])
    pb = choice_distribution(b["candidate_logits"], b["applicability_logit"])
torch.testing.assert_close(pb[:-1], pa[:-1][torch.tensor([2, 0, 1])], atol=1e-6, rtol=1e-5)
torch.testing.assert_close(pb[-1], pa[-1], atol=1e-6, rtol=1e-5)
test_b2 = B2StateQuery(dummy_embedding, hidden_size=32, d_model=24, heads=4, layers=2,
                       question_latents=4)
test_b2.eval()
with torch.no_grad():
    b2out = test_b2(state, qids, candidates)
    b2p = choice_distribution(b2out["candidate_logits"], b2out["applicability_logit"])
assert b2out["question_memory"].shape == (1, 4, 24)
assert b2p.shape[0] == len(candidates) + 1
test_b0 = B0CandidateConditioned(hidden_size=32, d_model=24).eval()
with torch.no_grad():
    b0out = test_b0.forward_features(torch.randn(3, 32))
    b0p = choice_distribution(b0out["candidate_logits"], b0out["applicability_logit"])
assert b0p.shape[0] == 4
print("B0/B1/B2 shape, probability, and candidate-permutation contracts passed.")


B0/B1/B2 shape, probability, and candidate-permutation contracts passed.


## 14. Feature-backed state episodes and state-balanced objective

One dataset item is one state plus all of its questions. The training objective is:

\[
L(\text{state})=\operatorname{mean}_{q\in\text{state}}L(q),\qquad
L(\text{batch})=\operatorname{mean}_{s\in\text{batch}}L(s)
\]

This prevents high-question-count states from dominating. Evidence supervision rewards attention mass on retained annotated evidence tokens; it is skipped when no evidence survives the declared truncation.


In [39]:
from safetensors.torch import load_file, save_file

def tokenize_query(tokenizer, text, max_length):
    return torch.tensor(tokenizer(
        text, add_special_tokens=False, truncation=True, max_length=max_length
    )["input_ids"], dtype=torch.long)

class StateEpisodeStore:
    def __init__(self, frames, feature_root, tokenizer):
        self.frames = frames
        self.feature_root = Path(feature_root)
        self.tokenizer = tokenizer
        manifest = json.loads((self.feature_root / "manifest.json").read_text())
        if manifest["corpus_manifest_sha256"] != CORPUS_LOCK["manifest_sha256"]:
            raise RuntimeError("Feature cache belongs to another corpus lock.")
        self.index = {r["state_id"]: r for r in manifest["state_files"]}
        self.control_index = {
            r["question_id"]: r for r in manifest.get("candidate_conditioned_control_files", [])
        }
        for record in {r["file"]: r for r in self.control_index.values()}.values():
            path = self.feature_root / record["file"]
            if not path.is_file() or sha256_file(path) != record["file_sha256"]:
                raise RuntimeError(f"Changed matched-control feature shard: {path.name}")
        self.states = frames["states"].set_index("state_id")
        self.questions = frames["questions"].set_index("question_id")
        self.options = {
            qid: g.sort_values("position").to_dict("records")
            for qid, g in frames["options"].groupby("question_id")
        }
        self.evidence = {
            qid: g.to_dict("records") for qid, g in frames["evidence"].groupby("question_id")
        } if len(frames["evidence"]) else {}

    def state_ids(self, split):
        mask = self.states["split"] == split
        if split == "train":
            mask &= self.states["eligible_for_training"].astype(bool)
        return sorted(self.states[mask].index.tolist())

    def load_state(self, state_id):
        record = self.index[state_id]
        tensors = load_file(self.feature_root / record["file"])
        return record, tensors

    def evidence_mask(self, qid, state_record, offsets):
        mask = torch.zeros(offsets.shape[0], dtype=torch.float32)
        effective = state_record["effective_state_text"]
        for ev in self.evidence.get(qid, []):
            snippet = ev.get("text")
            if not snippet:
                continue
            start = effective.find(snippet)
            if start < 0:
                continue
            end = start + len(snippet)
            overlap = (offsets[:, 0] < end) & (offsets[:, 1] > start)
            mask[overlap] = 1.0
        return mask

    def episode(self, state_id):
        state_record, tensors = self.load_state(state_id)
        qrows = self.frames["questions"][self.frames["questions"]["state_id"] == state_id]
        control_cache = {}
        items = []
        for q in qrows.to_dict("records"):
            opts = self.options[q["question_id"]]
            target = len(opts) if q["target_kind"] == "semantic_none" else next(
                i for i, o in enumerate(opts) if o["option_id"] == q["target_option_id"]
            )
            reason = q["applicability_reason"] if q["target_kind"] == "semantic_none" else "applicable"
            control_features = None
            control_record = self.control_index.get(q["question_id"])
            if control_record is not None:
                control_file = control_record["file"]
                if control_file not in control_cache:
                    control_cache[control_file] = load_file(self.feature_root / control_file)
                control_features = control_cache[control_file][control_record["key"]]
            items.append({
                "question": q,
                "options": opts,
                "question_ids": tokenize_query(self.tokenizer, q["instruction"], CFG["max_question_tokens"]),
                "candidate_ids": [
                    tokenize_query(self.tokenizer, o["description"], CFG["max_candidate_tokens"])
                    for o in opts
                ],
                "target": target,
                "reason_target": REASON_LABELS.index(reason),
                "evidence_mask": self.evidence_mask(q["question_id"], state_record, tensors["offsets"]),
                "control_features": control_features,
            })
        return tensors["hidden"], items

def question_loss(output, target, reason_target, evidence_mask,
                  evidence_weight=0.15, reason_weight=0.10):
    probs = choice_distribution(output["candidate_logits"], output["applicability_logit"])
    nll = -torch.log(probs[target].clamp_min(1e-8))
    reason = F.cross_entropy(output["reason_logits"].reshape(1, -1),
                             torch.tensor([reason_target], device=probs.device))
    evidence = torch.zeros((), device=probs.device)
    if evidence_mask.sum() > 0 and output["evidence_attention"] is not None:
        # heads x query_tokens x state_tokens; learned query/slot positions are first.
        attention = output["evidence_attention"][0].mean(dim=0)[0]
        mask = evidence_mask.to(device=attention.device, dtype=attention.dtype)
        evidence = -torch.log((attention * mask).sum().clamp_min(1e-8))
    total = nll + reason_weight * reason + evidence_weight * evidence
    return total, {"nll": nll.detach(), "reason": reason.detach(), "evidence": evidence.detach(),
                   "probabilities": probs.detach()}


## 15. GPU Stage B — train B0, B1, or B2 with the 4B model unloaded

This stage loads only the frozen input embedding and cached state sequences. It selects checkpoints on `development` NLL. Calibration and policy partitions are untouched during optimization.


In [41]:
# Keep probability and calibration math in FP32 even during BF16 model execution.
def choice_distribution(
    candidate_logits,
    applicability_logit,
    rank_temperature=1.0,
    app_temperature=1.0,
):
    device_type = candidate_logits.device.type

    with torch.autocast(device_type=device_type, enabled=False):
        logits = candidate_logits.float()
        app_logit = applicability_logit.float()

        rank_t = torch.as_tensor(
            rank_temperature,
            device=logits.device,
            dtype=torch.float32,
        )
        app_t = torch.as_tensor(
            app_temperature,
            device=logits.device,
            dtype=torch.float32,
        )

        if torch.any(rank_t <= 0) or torch.any(app_t <= 0):
            raise ValueError("Temperatures must be positive.")

        conditional = torch.softmax(logits / rank_t, dim=-1)
        applicability = torch.sigmoid(app_logit / app_t)

        probs = torch.cat([
            applicability * conditional,
            (1.0 - applicability).reshape(1),
        ])

        if not torch.isfinite(probs).all():
            raise FloatingPointError("Non-finite choice probabilities.")

        total = probs.sum()
        one = torch.ones((), device=probs.device, dtype=probs.dtype)

        if not torch.isclose(total.detach(), one, atol=1e-5, rtol=1e-5):
            raise AssertionError(
                f"Choice probabilities sum to {total.item():.8f}, not 1."
            )

        # Remove any remaining FP32 accumulation residue.
        return probs / total

print("FP32 probability calculation installed.")

FP32 probability calculation installed.


In [ ]:
TRAINING_COMPLETE = False
if not CFG["execute_gpu"]:
    print("Training skipped with GPU stage.")
else:
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(LOCAL_BUNDLE / "tokenizer", local_files_only=True)
    feature_root = DRIVE_RUN / "state_features"
    store = StateEpisodeStore(frames, feature_root, tokenizer)
    frozen_embedding = None

    if CFG["architecture"] == "B0":
        model = B0CandidateConditioned(
            hidden_size=MODEL_CONTRACT["hidden_size"], d_model=CFG["query_dim"]
        )
    else:
        frozen_embedding = load_file(
            feature_root / "qwen_input_embedding.bf16.safetensors"
        )["weight"]
        common = dict(
            hidden_size=MODEL_CONTRACT["hidden_size"], d_model=CFG["query_dim"],
            heads=CFG["query_heads"], layers=CFG["query_layers"],
            state_refiner_layers=CFG["state_refiner_layers"],
        )
    if CFG["architecture"] == "B1":
        model = B1StateQuery(frozen_embedding, **common)
    elif CFG["architecture"] in ("B2", "B2R"):
        if CFG["architecture"] == "B2R" and CFG["state_refiner_layers"] == 0:
            raise RuntimeError("B2R requires state_refiner_layers > 0")
        model = B2StateQuery(frozen_embedding, question_latents=CFG["question_latents"], **common)
    elif CFG["architecture"] != "B0":
        raise ValueError(CFG["architecture"])

    device = torch.device("cuda")
    model.to(device)
    parameters = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.AdamW(parameters, lr=CFG["learning_rate"], weight_decay=CFG["weight_decay"])
    train_ids = store.state_ids("train")
    dev_ids = store.state_ids("development")
    if not train_ids or not dev_ids:
        raise RuntimeError("Training and development states are required; use full mode if smoke split is sparse.")

    MODEL_IDENTITY = sha256_obj({
        "corpus": CORPUS_LOCK,
        "model_contract": MODEL_CONTRACT,
        "architecture": {k: CFG[k] for k in (
            "architecture", "query_dim", "query_heads", "query_layers",
            "question_latents", "state_refiner_layers", "max_state_tokens",
            "max_question_tokens", "max_candidate_tokens",
        )},
    })

    def run_split(state_ids, training):
        model.train(training)
        totals, rows = [], []
        if training:
            rng = random.Random(SEED + current_epoch)
            state_ids = list(state_ids); rng.shuffle(state_ids)
            optimizer.zero_grad(set_to_none=True)
        for state_n, state_id in enumerate(state_ids, 1):
            hidden, questions = store.episode(state_id)
            hidden = hidden.to(device)
            qlosses = []
            context = torch.enable_grad() if training else torch.no_grad()
            with context, torch.autocast("cuda", dtype=torch.bfloat16):
                # State projection/refinement executes once for every source state.
                memory = None if CFG["architecture"] == "B0" else model.project_state(hidden)
                for item in questions:
                    if CFG["architecture"] == "B0":
                        if item["control_features"] is None:
                            raise RuntimeError("Matched-control feature is missing.")
                        output = model.forward_features(item["control_features"].to(device))
                    else:
                        output = model.forward_memory(
                            memory,
                            item["question_ids"].to(device),
                            [x.to(device) for x in item["candidate_ids"]],
                        )
                    loss, detail = question_loss(
                        output, item["target"], item["reason_target"], item["evidence_mask"],
                        CFG["evidence_loss_weight"], CFG["reason_loss_weight"],
                    )
                    qlosses.append(loss)
                    rows.append({
                        "state_id": state_id,
                        "question_id": item["question"]["question_id"],
                        "loss": float(loss.detach().cpu()),
                        "nll": float(detail["nll"].cpu()),
                    })
                state_loss = torch.stack(qlosses).mean()
            totals.append(float(state_loss.detach().cpu()))
            if training:
                (state_loss / CFG["gradient_accumulation_states"]).backward()
                if state_n % CFG["gradient_accumulation_states"] == 0 or state_n == len(state_ids):
                    torch.nn.utils.clip_grad_norm_(parameters, 1.0)
                    optimizer.step(); optimizer.zero_grad(set_to_none=True)
        return float(np.mean(totals)), pd.DataFrame(rows)

    best = {"dev_nll": float("inf"), "epoch": None}
    history = []
    stale = 0
    checkpoint = LOCAL_RUN / "query_model_best.safetensors"
    for current_epoch in range(1, CFG["epochs"] + 1):
        train_loss, _ = run_split(train_ids, training=True)
        dev_loss, dev_rows = run_split(dev_ids, training=False)
        row = {"epoch": current_epoch, "train_state_loss": train_loss, "development_state_loss": dev_loss}
        history.append(row); print(row)
        if dev_loss < best["dev_nll"] - 1e-5:
            best = {"dev_nll": dev_loss, "epoch": current_epoch}
            save_file({
                k: v.detach().cpu() for k, v in model.state_dict().items()
                if k != "token_embedding.weight"
            }, checkpoint)
            stale = 0
        else:
            stale += 1
            if stale >= CFG["patience"]:
                break

    incompatible = model.load_state_dict(load_file(checkpoint), strict=False)
    expected_missing = {"token_embedding.weight"} if CFG["architecture"] != "B0" else set()
    if set(incompatible.missing_keys) != expected_missing or incompatible.unexpected_keys:
        raise RuntimeError(f"Unexpected checkpoint keys: {incompatible}")
    shutil.copy2(checkpoint, DRIVE_RUN / checkpoint.name)
    TRAINING_REPORT = {
        "schema": "opendecision-statequery-training/v1",
        "model_identity": MODEL_IDENTITY,
        "trainable_parameters": sum(p.numel() for p in parameters),
        "frozen_embedding_parameters": 0 if frozen_embedding is None else frozen_embedding.numel(),
        "history": history,
        "selected_epoch": best["epoch"],
        "selection_partition": "development",
        "final_opened": False,
    }
    for root in (LOCAL_RUN, DRIVE_RUN):
        (root / "TRAINING_REPORT.json").write_text(json.dumps(TRAINING_REPORT, indent=2))
    TRAINING_COMPLETE = True
    print(json.dumps(TRAINING_REPORT, indent=2))


## 16. Evaluation, calibration, and source/state-balanced reporting

Metrics are reported separately by source, primitive, family, semantic-none reason, and evidence-retention status. The notebook also emits a source-macro summary; it does not replace the component tables.


In [ ]:
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score

def row_brier(probabilities, target):
    p = np.asarray(probabilities, dtype=np.float64)
    y = np.zeros_like(p); y[target] = 1.0
    return float(np.sum((p - y) ** 2))

def row_ece(rows, bins=15):
    conf = np.asarray([max(r["probabilities"]) for r in rows], dtype=np.float64)
    correct = np.asarray([r["prediction_label"] == r["target_label"] for r in rows])
    total = 0.0
    edges = np.linspace(0, 1, bins + 1)
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (conf >= lo) & (conf < hi if hi < 1 else conf <= hi)
        if mask.any():
            total += mask.mean() * abs(correct[mask].mean() - conf[mask].mean())
    return float(total)

def metric_block(rows):
    if not rows:
        return {"n": 0}
    targets = [r["target_label"] for r in rows]
    pred = [r["prediction_label"] for r in rows]
    none_true = np.asarray([x == "__none__" for x in targets])
    none_pred = np.asarray([x == "__none__" for x in pred])
    tp = np.sum(none_true & none_pred); fn = np.sum(none_true & ~none_pred)
    return {
        "n": len(rows),
        "accuracy": float(accuracy_score(targets, pred)),
        "balanced_accuracy": float(balanced_accuracy_score(targets, pred)),
        "macro_f1": float(f1_score(targets, pred, average="macro", zero_division=0)),
        "nll": float(np.mean([-np.log(max(r["probabilities"][r["target"]], 1e-12)) for r in rows])),
        "brier": float(np.mean([row_brier(r["probabilities"], r["target"]) for r in rows])),
        "ece15": row_ece(rows),
        "semantic_none_recall": float(tp / max(1, tp + fn)),
    }

def state_bootstrap(rows, metric="accuracy", samples=1000, seed=17):
    if not rows:
        return {"low": None, "high": None}
    by_state = defaultdict(list)
    for row in rows: by_state[row["state_id"]].append(row)
    states = sorted(by_state)
    rng = np.random.default_rng(seed)
    values = []
    for _ in range(samples):
        draw = rng.choice(states, size=len(states), replace=True)
        sampled = [row for state_id in draw for row in by_state[state_id]]
        values.append(metric_block(sampled)[metric])
    return {"low": float(np.quantile(values, 0.025)), "high": float(np.quantile(values, 0.975))}

@dataclass
class Calibrator:
    rank_temperature: float = 1.0
    applicability_temperature: float = 1.0

    def probabilities(self, candidate_logits, applicability_logit):
        return choice_distribution(
            candidate_logits, applicability_logit,
            self.rank_temperature, self.applicability_temperature,
        )

def collect_predictions(model, store, split, calibrator=Calibrator()):
    model.eval(); rows = []
    device = next(model.parameters()).device
    with torch.no_grad():
        for state_id in store.state_ids(split):
            hidden, questions = store.episode(state_id); hidden = hidden.to(device)
            source = store.states.loc[state_id, "source"]
            with torch.autocast("cuda", dtype=torch.bfloat16):
                memory = None if CFG["architecture"] == "B0" else model.project_state(hidden)
            for item in questions:
                with torch.autocast("cuda", dtype=torch.bfloat16):
                    if CFG["architecture"] == "B0":
                        if item["control_features"] is None:
                            raise RuntimeError("Matched-control feature is missing for evaluation.")
                        output = model.forward_features(item["control_features"].to(device))
                    else:
                        output = model.forward_memory(
                            memory, item["question_ids"].to(device),
                            [x.to(device) for x in item["candidate_ids"]]
                        )
                    probs = calibrator.probabilities(
                        output["candidate_logits"].float(), output["applicability_logit"].float()
                    )
                option_ids = [o["option_id"] for o in item["options"]] + ["__none__"]
                prediction = int(probs.argmax().cpu())
                rows.append({
                    "state_id": state_id, "question_id": item["question"]["question_id"],
                    "source": source, "primitive": item["question"]["primitive"],
                    "family": item["question"]["family"], "target": item["target"],
                    "target_label": option_ids[item["target"]],
                    "prediction_label": option_ids[prediction],
                    "option_ids": option_ids,
                    "candidate_logits": output["candidate_logits"].float().cpu().tolist(),
                    "applicability_logit": float(output["applicability_logit"].float().cpu()),
                    "probabilities": probs.cpu().tolist(),
                })
    return rows

def reference_predictions_for_split(store, split):
    path = DRIVE_RUN / "reference_nonfinal_predictions.parquet"
    if not path.is_file():
        return []
    raw = pd.read_parquet(path).set_index("question_id")
    rows = []
    for state_id in store.state_ids(split):
        source = store.states.loc[state_id, "source"]
        qrows = store.frames["questions"][store.frames["questions"]["state_id"] == state_id]
        for q in qrows.to_dict("records"):
            qid = q["question_id"]
            if qid not in raw.index:
                continue
            record = raw.loc[qid]
            option_rows = store.options[qid]
            option_ids = [o["option_id"] for o in option_rows] + ["__none__"]
            option_probs = json.loads(record["option_probabilities_json"])
            none_probability = record["none_probability"]
            if pd.isna(none_probability):
                none_probability = max(0.0, 1.0 - sum(float(option_probs.get(k, 0.0)) for k in option_ids[:-1]))
            probs = [float(option_probs.get(k, 0.0)) for k in option_ids[:-1]] + [float(none_probability)]
            target = len(option_rows) if q["target_kind"] == "semantic_none" else next(
                i for i, o in enumerate(option_rows) if o["option_id"] == q["target_option_id"]
            )
            prediction = int(np.argmax(probs))
            rows.append({
                "state_id": state_id, "question_id": qid, "source": source,
                "primitive": q["primitive"], "family": q["family"], "target": target,
                "target_label": option_ids[target], "prediction_label": option_ids[prediction],
                "option_ids": option_ids, "probabilities": probs,
            })
    return rows

def by_source_metrics(rows):
    return {
        source: metric_block([r for r in rows if r["source"] == source])
        for source in sorted(set(r["source"] for r in rows))
    }

def source_macro(metrics):
    keys = ("accuracy", "balanced_accuracy", "macro_f1", "nll", "brier", "ece15", "semantic_none_recall")
    return {k: float(np.mean([v[k] for v in metrics.values() if k in v])) for k in keys}

def fit_temperatures(rows):
    if not rows:
        raise ValueError("Calibration rows are empty; check the split audit before fitting temperatures.")
    log_rank = torch.zeros((), requires_grad=True)
    log_app = torch.zeros((), requires_grad=True)
    optimizer = torch.optim.LBFGS([log_rank, log_app], lr=0.2, max_iter=80, line_search_fn="strong_wolfe")
    def closure():
        optimizer.zero_grad(); losses = []
        for row in rows:
            p = choice_distribution(
                torch.tensor(row["candidate_logits"]), torch.tensor(row["applicability_logit"]),
                log_rank.exp(), log_app.exp(),
            )
            losses.append(-torch.log(p[row["target"]].clamp_min(1e-12)))
        loss = torch.stack(losses).mean(); loss.backward(); return loss
    optimizer.step(closure)
    return Calibrator(float(log_rank.exp().detach()), float(log_app.exp().detach()))

def entropy_confidence(probabilities):
    p = np.asarray(probabilities, dtype=np.float64)
    p = p / p.sum()
    positive = p > 0
    entropy = -np.sum(p[positive] * np.log(p[positive]))
    return float(1.0 - entropy / np.log(len(p))) if len(p) > 1 else 1.0

def policy_metrics(rows, threshold):
    accepted = [r for r in rows if entropy_confidence(r["probabilities"]) >= threshold]
    wrong = sum(r["prediction_label"] != r["target_label"] for r in accepted)
    coverage = len(accepted) / max(1, len(rows))
    accepted_error = wrong / len(accepted) if accepted else None
    cost = (
        CFG["policy_wrong_cost"] * wrong
        + CFG["policy_review_cost"] * (len(rows) - len(accepted))
    ) / max(1, len(rows))
    return {"threshold": float(threshold), "coverage": float(coverage),
            "accepted_error": None if accepted_error is None else float(accepted_error),
            "accepted": len(accepted), "wrong_accepted": int(wrong), "cost": float(cost)}

def select_policy(rows):
    candidates = sorted({0.0, 1.0, *[entropy_confidence(r["probabilities"]) for r in rows]})
    evaluated = [policy_metrics(rows, threshold) for threshold in candidates]
    return min(evaluated, key=lambda x: (x["cost"], -x["coverage"], -x["threshold"]))

if TRAINING_COMPLETE:
    raw_cal = collect_predictions(model, store, "calibration_fit")
    calibrator = fit_temperatures(raw_cal)
    gate_rows = collect_predictions(model, store, "calibration_gate", calibrator)
    policy_rows = collect_predictions(model, store, "policy_development", calibrator)
    reference_gate_rows = reference_predictions_for_split(store, "calibration_gate")
    custom_by_source = by_source_metrics(gate_rows)
    reference_by_source = by_source_metrics(reference_gate_rows)
    custom_macro = source_macro(custom_by_source)
    reference_macro = source_macro(reference_by_source) if reference_by_source else {}
    macro_delta = {
        key: custom_macro[key] - reference_macro[key]
        for key in custom_macro.keys() & reference_macro.keys()
    }
    selected_policy = select_policy(policy_rows)
    gate_policy = policy_metrics(gate_rows, selected_policy["threshold"])
    REPORT = {
        "calibrator": asdict(calibrator),
        "calibration_gate_diagnostic_pooled": metric_block(gate_rows),
        "policy_development": metric_block(policy_rows),
        "custom_by_source": custom_by_source,
        "reference_by_source": reference_by_source,
        "custom_source_macro": custom_macro,
        "reference_source_macro": reference_macro,
        "custom_minus_reference_source_macro": macro_delta,
        "policy_selected_on_policy_development": selected_policy,
        "policy_applied_to_calibration_gate": gate_policy,
        "state_bootstrap_accuracy": state_bootstrap(gate_rows),
        "final_opened": False,
    }
    for root in (LOCAL_RUN, DRIVE_RUN):
        (root / "NONFINAL_EVALUATION.json").write_text(json.dumps(REPORT, indent=2))
    print(json.dumps(REPORT, indent=2))
else:
    print("Evaluation waits for a completed training stage.")


## 17. Freeze model selection before final evaluation

The lock is created from non-final evidence only. It is blocked while any promotion limit is unset. An exploratory smoke run may still export its checkpoint, but it cannot claim a promoted model.


In [ ]:
MODEL_LOCK = None
if TRAINING_COMPLETE:
    missing_gates = [k for k, v in CFG["promotion_gates"].items() if v is None]
    gate_checks = {}
    if not missing_gates:
        delta = REPORT["custom_minus_reference_source_macro"]
        policy_gate = REPORT["policy_applied_to_calibration_gate"]
        gate_checks = {
            "source_macro_accuracy": delta["accuracy"] >= CFG["promotion_gates"]["source_macro_accuracy_delta_min"],
            "source_macro_nll": delta["nll"] <= CFG["promotion_gates"]["source_macro_nll_delta_max"],
            "source_macro_ece": delta["ece15"] <= CFG["promotion_gates"]["source_macro_ece_delta_max"],
            "semantic_none_recall": delta["semantic_none_recall"] >= CFG["promotion_gates"]["semantic_none_recall_delta_min"],
            "accepted_error": (
                policy_gate["accepted_error"] is not None
                and policy_gate["accepted_error"] <= CFG["promotion_gates"]["accepted_error_max"]
            ),
            "coverage": policy_gate["coverage"] >= CFG["promotion_gates"]["minimum_coverage"],
        }
    status = (
        "exploratory_locked_gates_unset" if missing_gates
        else "locked_before_final" if all(gate_checks.values())
        else "locked_not_promoted"
    )
    MODEL_LOCK = {
        "schema": "opendecision-statequery-model-lock/v1",
        "run_id": RUN_ID,
        "model_identity": MODEL_IDENTITY,
        "architecture": CFG["architecture"],
        "checkpoint_sha256": sha256_file(DRIVE_RUN / "query_model_best.safetensors"),
        "corpus_lock": CORPUS_LOCK,
        "calibration": asdict(calibrator),
        "nonfinal_report_sha256": sha256_file(DRIVE_RUN / "NONFINAL_EVALUATION.json"),
        "promotion_gates": CFG["promotion_gates"],
        "status": status,
        "missing_gates": missing_gates,
        "gate_checks": gate_checks,
        "final_opened": False,
    }
    for root in (LOCAL_RUN, DRIVE_RUN):
        (root / "MODEL_SELECTION_LOCK.json").write_text(json.dumps(MODEL_LOCK, indent=2))
    print(json.dumps(MODEL_LOCK, indent=2))
else:
    print("No model-selection lock until training and non-final evaluation complete.")


## 18. Final evaluation — intentionally closed by default

This cell is a one-way study event, not a convenience preview. It requires:

- full (not smoke) corpus mode;
- all promotion limits set;
- a lock created before final predictions;
- `CFG["open_final"] = True` set deliberately after reviewing the lock.

Re-running a completed final evaluation returns the saved result; it does not reselect the model.


In [ ]:
if CFG["open_final"]:
    if CFG["run_mode"] != "full":
        raise RuntimeError("Final evaluation is forbidden in smoke mode.")
    if MODEL_LOCK is None or MODEL_LOCK["status"] != "locked_before_final":
        raise RuntimeError("A complete non-final model lock with reviewed gates is required.")
    if CFG["architecture"] == "B0":
        raise RuntimeError(
            "B0 final candidate-conditioned features were intentionally not prefetched. "
            "Add a post-lock, final-only reference feature pass before opening B0 final results."
        )
    final_path = DRIVE_RUN / "FINAL_EVALUATION.json"
    if final_path.exists():
        FINAL_REPORT = json.loads(final_path.read_text())
        print("Loaded existing immutable final result.")
    else:
        final_rows = collect_predictions(model, store, "final", calibrator)
        FINAL_REPORT = {
            "schema": "opendecision-statequery-final/v1",
            "run_id": RUN_ID,
            "model_lock_sha256": sha256_file(DRIVE_RUN / "MODEL_SELECTION_LOCK.json"),
            "metrics": metric_block(final_rows),
            "by_source": {source: metric_block([r for r in final_rows if r["source"] == source])
                          for source in sorted(set(r["source"] for r in final_rows))},
            "state_bootstrap_accuracy": state_bootstrap(final_rows),
            "opened_once": True,
            "no_reselection_after_final": True,
        }
        final_path.write_text(json.dumps(FINAL_REPORT, indent=2))
        shutil.copy2(final_path, LOCAL_RUN / final_path.name)
    print(json.dumps(FINAL_REPORT, indent=2))
else:
    print("Final split remains closed (expected).")


## 19. Rust-facing export contract

The export uses only standard operators (`Linear`, `LayerNorm`, multi-head attention, GELU, softmax, sigmoid). Golden fixtures come from `policy_development`, never `final`.

Planned bundle:

```text
opendecision-statequery-v1/
├── manifest.json
├── query_config.json
├── query_model.safetensors
├── calibration.json
├── criteria_registry.json
├── golden_inputs.safetensors
├── golden_intermediates.safetensors
└── golden_outputs.json
```

The runtime should model this as immutable `StaticStateMemory`, not force it into Qwen's mutable continuation-state abstraction.


In [ ]:
def export_statequery_bundle():
    if not TRAINING_COMPLETE or MODEL_LOCK is None:
        print("Export skipped: no trained, locked query model.")
        return None
    export = DRIVE_RUN / "opendecision-statequery-v1"
    export.mkdir(exist_ok=False)
    shutil.copy2(DRIVE_RUN / "query_model_best.safetensors", export / "query_model.safetensors")
    shutil.copy2(CORPUS_DRIVE / "criteria_registry.json", export / "criteria_registry.json")
    (export / "query_config.json").write_text(json.dumps({
        "schema": "opendecision-statequery-config/v1",
        "architecture": CFG["architecture"],
        "hidden_size": MODEL_CONTRACT["hidden_size"],
        "query_dim": CFG["query_dim"],
        "query_heads": CFG["query_heads"],
        "query_layers": CFG["query_layers"],
        "question_latents": CFG["question_latents"],
        "state_refiner_layers": CFG["state_refiner_layers"],
        "probability_contract": "P(candidate)=a*softmax(score); P(__none__)=1-a",
    }, indent=2))
    (export / "calibration.json").write_text(json.dumps(asdict(calibrator), indent=2))
    # Golden tensor emission is intentionally a separate explicit step after selection.
    (export / "GOLDEN_FIXTURES_PENDING.md").write_text(
        "Generate policy-development-only golden inputs, state projection, question memory, "
        "candidate logits, applicability logit, and final probabilities before Rust parity.\n"
    )
    files = {p.name: sha256_file(p) for p in export.iterdir() if p.is_file()}
    manifest = {
        "schema": "opendecision-statequery-bundle/v1",
        "run_id": RUN_ID,
        "model_identity": MODEL_IDENTITY,
        "base_model": MODEL_CONTRACT,
        "corpus_lock": CORPUS_LOCK,
        "model_selection_lock_sha256": sha256_file(DRIVE_RUN / "MODEL_SELECTION_LOCK.json"),
        "files": files,
        "final_evidence_included": False,
        "production_ready": False,
    }
    (export / "manifest.json").write_text(json.dumps(manifest, indent=2))
    return export

EXPORTED_BUNDLE = export_statequery_bundle()
print("Export:", EXPORTED_BUNDLE)


## 20. Scope and next edits

Implemented in this initial notebook:

- exact selected-profile/bundle verification using Phase 3B conventions;
- state-centric schema and immutable corpus lock;
- ContractNLI, QASPER, MultiRC, SQuAD 2.0, and MASSIVE adapters;
- group-safe six-stage splits and leakage audits;
- predeclared semantic-ID-aligned perturbations;
- frozen 4B reference prediction + state-feature cache path;
- B1 direct and B2 question-factorized query modules, optional state refiner;
- state-balanced training, evidence and applicability-reason auxiliary losses;
- calibration, task/source-specific metrics, state bootstrap, model lock, closed final gate;
- Rust-facing bundle skeleton.

Still intentionally open:

1. review and sign the exact dataset/license manifest before a distributable-weights run;
2. add the QuALITY and SummEval adapters after per-source license/field validation;
3. add a proper connected-component/near-duplicate audit beyond exact text and upstream IDs;
4. fill numerical promotion gates before any final evaluation;
5. add the post-lock final-only B0 feature pass, then keep final predictions immutable;
6. implement policy-development golden tensor export and reload parity;
7. run B0, B1-2, B1-4, B2, and B2R under the same locked corpus;
8. only after the 4B representation gate, add Qwen3.5-2B, Qwen3.5-0.8B, ModernBERT, and DeBERTa comparisons;
9. add teacher distillation only after the human-label architecture test.
